# Análisis de la Corrida de Prueba

**Proyecto Final — Arquitectura de Grandes Volúmenes de Datos (ITAM, Primavera 2026)**

Este notebook analiza y visualiza los resultados de la **corrida de prueba** del pipeline.
La prueba se ejecutó con ~6 minutos de acumulación de datos (el pipeline final usará 20+ minutos).

**Objetivo:** Verificar que el pipeline completo funciona de punta a punta y entender
las características de los datos antes de la ejecución final.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import json
import os

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

# Rutas a los datos de la prueba
TABLEAU_DIR = os.path.join('output', 'tableau')
METRICS_DIR = os.path.join('output', 'metrics')

## 1. Carga de Datos

In [ ]:
stats = pd.read_csv(os.path.join(TABLEAU_DIR, 'window_statistics.csv'))
labeled = pd.read_csv(os.path.join(TABLEAU_DIR, 'labeled_data.csv'))
predictions = pd.read_csv(os.path.join(TABLEAU_DIR, 'predictions.csv'))

# Convertir timestamps
for df in [stats, labeled, predictions]:
    df['window_start'] = pd.to_datetime(df['window_start_str'])
    df['window_end'] = pd.to_datetime(df['window_end_str'])

print(f'Estadísticos por ventana:  {len(stats)} filas')
print(f'Datos etiquetados (K-Means): {len(labeled)} filas')
print(f'Predicciones (LogReg):     {len(predictions)} filas')
print(f'Símbolos: {sorted(stats["symbol"].unique())}')

## 2. Estadísticos por Ventana — Primera Tanda de Streaming

Estos son los estadísticos calculados en tiempo real durante el Momento 1 del pipeline.
Cada punto representa una ventana de 1 minuto (con slide de 30s) para un símbolo.

In [ ]:
# Ventanas por símbolo
windows_per_symbol = stats.groupby('symbol').size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras: cantidad de ventanas por símbolo
colors = plt.cm.Set2(range(len(windows_per_symbol)))
windows_per_symbol.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Ventanas de tiempo por símbolo')
axes[0].set_ylabel('Cantidad de ventanas')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

# Barras: trades promedio por ventana por símbolo
avg_trades = stats.groupby('symbol')['trade_count'].mean().sort_values(ascending=False)
avg_trades.plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_title('Trades promedio por ventana (por símbolo)')
axes[1].set_ylabel('Trades por ventana')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('fig_ventanas_por_simbolo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Serie temporal del precio promedio para los 3 pares principales
top_symbols = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']

fig, axes = plt.subplots(len(top_symbols), 1, figsize=(14, 10), sharex=True)

for i, sym in enumerate(top_symbols):
    sym_data = stats[stats['symbol'] == sym].sort_values('window_start')
    ax = axes[i]
    
    ax.plot(sym_data['window_start'], sym_data['price_avg'], 
            linewidth=1.5, label='Precio promedio', color='steelblue')
    ax.fill_between(sym_data['window_start'], 
                    sym_data['price_min'], sym_data['price_max'],
                    alpha=0.2, color='steelblue', label='Rango (min-max)')
    ax.set_title(f'{sym} — Precio por ventana de tiempo')
    ax.set_ylabel('Precio (USDT)')
    ax.legend(loc='upper right', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

axes[-1].set_xlabel('Hora')
plt.tight_layout()
plt.savefig('fig_serie_temporal_precios.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. K-Means — Detección de Anomalías (Momento 1)

K-Means agrupó las ventanas en 3 clusters (elegido por Silhouette Score).
Los puntos con distancia al centroide > percentil 95 se etiquetaron como anómalos.

**Nota de prueba:** Con solo ~6 minutos de datos se obtuvieron 24 anomalías de 430 ventanas (5.6%).
En la ejecución final con 20 minutos se espera tener ~70 anomalías, lo que dará
un mejor balance para el modelo supervisado.

In [ ]:
# Distribución de etiquetas
label_counts = labeled['label'].value_counts()
print('Distribución de etiquetas K-Means:')
print(label_counts)
print(f'\nPorcentaje de anomalías: {label_counts.get("anomalo", 0) / len(labeled) * 100:.1f}%')

In [ ]:
# Scatter: volatilidad vs retorno, coloreado por etiqueta
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

normal = labeled[labeled['label'] == 'normal']
anomalo = labeled[labeled['label'] == 'anomalo']

# Volatilidad vs Retorno
axes[0].scatter(normal['pct_return'], normal['volatility'],
                alpha=0.5, s=30, label='Normal', color='steelblue')
axes[0].scatter(anomalo['pct_return'], anomalo['volatility'],
                alpha=0.8, s=60, label='Anómalo', color='crimson', marker='X')
axes[0].set_xlabel('Retorno porcentual')
axes[0].set_ylabel('Volatilidad')
axes[0].set_title('Volatilidad vs Retorno')
axes[0].legend()

# Price speed vs Volume intensity
axes[1].scatter(normal['volume_intensity'], normal['price_speed'],
                alpha=0.5, s=30, label='Normal', color='steelblue')
axes[1].scatter(anomalo['volume_intensity'], anomalo['price_speed'],
                alpha=0.8, s=60, label='Anómalo', color='crimson', marker='X')
axes[1].set_xlabel('Intensidad de volumen')
axes[1].set_ylabel('Velocidad de cambio del precio')
axes[1].set_title('Velocidad de precio vs Intensidad de volumen')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_kmeans_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribución de distancias al centroide con umbral de anomalía
fig, ax = plt.subplots(figsize=(14, 5))

ax.hist(labeled['distance_to_centroid'], bins=40, color='steelblue',
        alpha=0.7, edgecolor='white', label='Todas las ventanas')

# Umbral de anomalía (p95)
threshold = labeled[labeled['label'] == 'anomalo']['distance_to_centroid'].min()
ax.axvline(threshold, color='crimson', linewidth=2, linestyle='--',
           label=f'Umbral p95 = {threshold:.2f}')

# Marcar los anómalos
ax.hist(anomalo['distance_to_centroid'], bins=20, color='crimson',
        alpha=0.5, edgecolor='white', label='Anómalos')

ax.set_xlabel('Distancia al centroide')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de distancias al centroide — Umbral de anomalía (percentil 95)')
ax.legend()

plt.tight_layout()
plt.savefig('fig_distancia_centroide.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Clusters por símbolo
cluster_by_symbol = labeled.groupby(['symbol', 'prediction']).size().unstack(fill_value=0)

cluster_by_symbol.plot(kind='bar', stacked=True, figsize=(14, 5),
                       colormap='Set2', edgecolor='white')
plt.title('Distribución de clusters K-Means por símbolo')
plt.xlabel('')
plt.ylabel('Cantidad de ventanas')
plt.legend(title='Cluster', bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('fig_clusters_por_simbolo.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Logistic Regression — Modelo Supervisado (Momento 2)

El modelo se entrenó con los datos etiquetados por K-Means (80% train, 20% test).

**Resultados de la prueba:**

| Métrica | Valor |
|---|---|
| Accuracy | 94.57% |
| Precision (weighted) | 89.43% |
| Recall (weighted) | 94.57% |
| F1-score (weighted) | 91.92% |
| AUC-ROC | 1.0000 |

**Limitación importante:** Con solo 5 anomalías en el set de test, el modelo las clasificó
todas como "normal". El AUC-ROC perfecto (1.0) indica que las probabilidades sí separan
las clases correctamente — el problema es el umbral de decisión (0.5) que es conservador
dado el desbalance de clases (5.6% anomalías). Con más datos en la ejecución final,
habrá suficientes anomalías para que el modelo aprenda un umbral útil.

**Coeficientes:** price_speed (0.53) y volatilidad (0.43) son los features más influyentes,
lo cual tiene sentido: movimientos rápidos y alta volatilidad son los indicadores más
fuertes de comportamiento anómalo en el mercado.

## 5. Inferencia en Streaming — Segunda Tanda (Momento 3)

El modelo entrenado se aplicó a datos nuevos en tiempo real. A diferencia del set de test
del Momento 2, aquí el modelo **sí detectó anomalías**, lo que sugiere que los datos nuevos
incluían ventanas con comportamiento más extremo.

In [ ]:
# Distribución de predicciones en la segunda tanda
pred_counts = predictions['label_predicted'].value_counts()
print('Predicciones en la segunda tanda de streaming:')
print(pred_counts)
print(f'\nPorcentaje de anomalías detectadas: {pred_counts.get("anomalo", 0) / len(predictions) * 100:.1f}%')

In [ ]:
# Distribución de probabilidad de anomalía
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de prob_anomalo
axes[0].hist(predictions['prob_anomalo'], bins=40, color='steelblue',
             alpha=0.7, edgecolor='white')
axes[0].axvline(0.5, color='crimson', linewidth=2, linestyle='--',
                label='Umbral 0.5')
axes[0].set_xlabel('P(anómalo)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de probabilidad de anomalía')
axes[0].legend()

# Anomalías por símbolo
anomalies_by_symbol = predictions[predictions['label_predicted'] == 'anomalo'].groupby('symbol').size()
total_by_symbol = predictions.groupby('symbol').size()
pct_anomalies = (anomalies_by_symbol / total_by_symbol * 100).fillna(0).sort_values(ascending=False)

pct_anomalies.plot(kind='bar', ax=axes[1], color='crimson', alpha=0.7, edgecolor='white')
axes[1].set_title('Porcentaje de anomalías detectadas por símbolo')
axes[1].set_ylabel('% anómalo')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('fig_predicciones_inference.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Serie temporal de precio con anomalías marcadas (BTCUSDT)
btc_pred = predictions[predictions['symbol'] == 'BTCUSDT'].sort_values('window_start')

if len(btc_pred) > 0:
    fig, ax = plt.subplots(figsize=(14, 5))
    
    btc_normal = btc_pred[btc_pred['label_predicted'] == 'normal']
    btc_anomalo = btc_pred[btc_pred['label_predicted'] == 'anomalo']
    
    ax.plot(btc_pred['window_start'], btc_pred['price_avg'],
            linewidth=1.5, color='steelblue', label='Precio promedio')
    ax.fill_between(btc_pred['window_start'],
                    btc_pred['price_min'], btc_pred['price_max'],
                    alpha=0.15, color='steelblue')
    
    if len(btc_anomalo) > 0:
        ax.scatter(btc_anomalo['window_start'], btc_anomalo['price_avg'],
                   color='crimson', s=80, zorder=5, marker='X',
                   label=f'Anómalo ({len(btc_anomalo)} ventanas)')
    
    ax.set_title('BTCUSDT — Precio con detección de anomalías (segunda tanda)')
    ax.set_xlabel('Hora')
    ax.set_ylabel('Precio (USDT)')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    
    plt.tight_layout()
    plt.savefig('fig_btc_anomalias.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No hay datos de BTCUSDT en las predicciones.')

## 6. Comparación de Features entre Momentos

Comparamos la distribución de features entre la primera tanda (K-Means) y la segunda
tanda (inferencia) para verificar que los datos son comparables.

In [ ]:
features = ['pct_return', 'volatility', 'volume_intensity', 'price_speed']
feature_labels = ['Retorno %', 'Volatilidad', 'Intensidad volumen', 'Velocidad precio']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (feat, label) in enumerate(zip(features, feature_labels)):
    ax = axes[idx // 2][idx % 2]
    
    ax.hist(labeled[feat], bins=30, alpha=0.5, color='steelblue',
            label='Momento 1 (K-Means)', density=True, edgecolor='white')
    ax.hist(predictions[feat], bins=30, alpha=0.5, color='coral',
            label='Momento 3 (Inferencia)', density=True, edgecolor='white')
    
    ax.set_title(label)
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=9)

plt.suptitle('Distribución de features — Momento 1 vs Momento 3', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig_features_comparacion.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Hardware de la Prueba

In [ ]:
hw_path = os.path.join(METRICS_DIR, 'hardware_info.json')
with open(hw_path, 'r') as f:
    hw = json.load(f)

print('Máquina de prueba:')
print(f'  OS:    {hw["os"]["system"]} {hw["os"]["release"]}')
print(f'  CPU:   {hw["cpu"]["model"]} — {hw["cpu"]["logical_cores"]} cores')
print(f'  RAM:   {hw["ram_gb"]} GB')
print(f'  GPU:   {"No detectada" if hw["gpu"] is None else hw["gpu"]["model"]}')
print(f'  Disco: {hw["disk"]["free_gb"]} GB libres de {hw["disk"]["total_gb"]} GB')
print(f'  Spark: {hw["software"]["pyspark_pip"]}')
print(f'  Kafka: {hw["software"]["kafka"]}')

## 8. Conclusiones de la Prueba

### Lo que se validó

1. **El pipeline completo funciona de punta a punta:** desde la captura de trades en tiempo real
   de Binance hasta las predicciones del modelo supervisado, pasando por Kafka, Spark Structured
   Streaming, K-Means y Logistic Regression.

2. **K-Means produce clusters bien separados:** Silhouette Score de 0.8245 con k=3, lo que indica
   que los features elegidos (retorno, volatilidad, intensidad de volumen, velocidad de precio)
   capturan patrones reales en los datos del mercado.

3. **El modelo supervisado generaliza:** en la segunda tanda de streaming, Logistic Regression
   detectó 22 anomalías de 254 ventanas nuevas (8.7%), demostrando que el modelo transfiere
   el conocimiento aprendido del K-Means a datos no vistos.

### Áreas de mejora para la ejecución final

1. **Más datos de acumulación (20+ minutos):** con solo 430 ventanas, el set de test tuvo
   5 anomalías — insuficiente para métricas de evaluación robustas. Con 1400+ ventanas
   se espera tener ~70 anomalías y un split más significativo.

2. **Captura de métricas de Spark UI en vivo:** la Fase 5 debe ejecutarse en paralelo con
   las fases de streaming para capturar métricas reales de ejecución.

3. **Comparación entre máquinas:** esta prueba solo se ejecutó en la Mac (Máquina A).
   La ejecución final incluirá la máquina Linux con GPU para la comparación de arquitecturas.

4. **Visualización en Tableau:** los CSVs están listos. Falta construir los dashboards
   animados con la serie temporal de precios y las métricas de rendimiento.